In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent

cases = pd.read_csv(
    ROOT / "data/synthetic/disease_cases.csv",
    parse_dates=["year_month"]
)

thresholds = pd.read_csv(
    ROOT / "data/processed/recovery_thresholds.csv"
)

# Use the original baseline scenario first.
baseline = cases[
    cases["scenario_id"] == "baseline"
].copy()

baseline["calendar_month"] = baseline["year_month"].dt.month

thresholds["calendar_month"] = (
    thresholds["calendar_month"].astype(int)
)

recovery_data = baseline.merge(
    thresholds[
        ["district_id", "disease_id",
         "calendar_month", "threshold_cases"]
    ],
    on=["district_id", "disease_id", "calendar_month"],
    how="left",
    validate="many_to_one"
)

recovery_data["above_threshold"] = (
    recovery_data["cases"] > recovery_data["threshold_cases"]
)

recovery_data = recovery_data.sort_values(
    ["district_id", "disease_id", "year_month"]
).reset_index(drop=True)

print("=== RECOVERY ENGINE INPUT VERIFICATION ===")
print("Total records:", len(recovery_data))
print(
    "Missing thresholds:",
    recovery_data["threshold_cases"].isna().sum()
)
print("Diseases:", sorted(recovery_data["disease_id"].unique()))
print("Districts:", sorted(recovery_data["district_id"].unique()))

print("\nSample:")
display(
    recovery_data[
        ["year_month", "district_id", "disease_id",
         "cases", "threshold_cases", "above_threshold"]
    ].head(10)
)


=== RECOVERY ENGINE INPUT VERIFICATION ===
Total records: 1200
Missing thresholds: 0
Diseases: ['dengue', 'malaria']
Districts: ['cbe', 'chn', 'mdu', 'tnj', 'try']

Sample:


,year_month,district_id,disease_id,cases,threshold_cases,above_threshold
0,2015-01-01,cbe,dengue,120,127.2,False
1,2015-02-01,cbe,dengue,106,106.8,False
2,2015-03-01,cbe,dengue,68,81.6,False
3,2015-04-01,cbe,dengue,67,61.2,True
4,2015-05-01,cbe,dengue,62,48.0,True
5,2015-06-01,cbe,dengue,56,61.2,False
6,2015-07-01,cbe,dengue,81,73.2,True
7,2015-08-01,cbe,dengue,84,100.8,False
8,2015-09-01,cbe,dengue,175,109.2,True
9,2015-10-01,cbe,dengue,163,153.6,True


In [2]:

# Detect outbreaks and sustained recovery

SUSTAINED_MONTHS = 3

def detect_recovery_episodes(group, sustained_months=3):
    group = group.sort_values("year_month").reset_index(drop=True)

    episodes = []
    active = False
    onset = None
    below_streak = 0
    interruptions = 0
    had_below_month = False

    for _, row in group.iterrows():
        date = row["year_month"]
        above = bool(row["above_threshold"])

        if not active:
            if above:
                active = True
                onset = date
                below_streak = 0
                interruptions = 0
                had_below_month = False
            continue

        if above:
            if below_streak > 0:
                interruptions += 1

            below_streak = 0

        else:
            below_streak += 1
            had_below_month = True

            if below_streak >= sustained_months:
                recovery_date = date

                recovery_months = (
                    (recovery_date.year - onset.year) * 12
                    + recovery_date.month - onset.month
                )

                episodes.append({
                    "district_id": group["district_id"].iloc[0],
                    "disease_id": group["disease_id"].iloc[0],
                    "outbreak_onset": onset,
                    "recovery_date": recovery_date,
                    "recovery_months": recovery_months,
                    "recovery_interruptions": interruptions,
                    "status": "recovered"
                })

                active = False
                onset = None
                below_streak = 0
                interruptions = 0
                had_below_month = False

    # An outbreak still active at the end of the dataset
    if active:
        episodes.append({
            "district_id": group["district_id"].iloc[0],
            "disease_id": group["disease_id"].iloc[0],
            "outbreak_onset": onset,
            "recovery_date": pd.NaT,
            "recovery_months": np.nan,
            "recovery_interruptions": interruptions,
            "status": "unresolved"
        })

    return episodes


all_episodes = []

for (district, disease), group in recovery_data.groupby(
    ["district_id", "disease_id"]
):
    all_episodes.extend(
        detect_recovery_episodes(
            group,
            sustained_months=SUSTAINED_MONTHS
        )
    )

recovery_results = pd.DataFrame(all_episodes)

print("=== RECOVERY EPISODE RESULTS ===")
print("Total episodes:", len(recovery_results))

if not recovery_results.empty:
    print("\nRecovery status:")
    print(recovery_results["status"].value_counts())

    print("\nSample episodes:")
    display(recovery_results.head(15))

    print("\nSummary by disease:")
    display(
        recovery_results.groupby("disease_id").agg(
            episodes=("status", "size"),
            recovered=("status", lambda x: (x == "recovered").sum()),
            unresolved=("status", lambda x: (x == "unresolved").sum()),
            mean_recovery_months=("recovery_months", "mean"),
            total_interruptions=("recovery_interruptions", "sum")
        ).round(2)
    )
else:
    print("No outbreak episodes detected.")


=== RECOVERY EPISODE RESULTS ===
Total episodes: 92

Recovery status:
status
recovered     88
unresolved     4
Name: count, dtype: int64

Sample episodes:


,district_id,disease_id,outbreak_onset,recovery_date,recovery_months,recovery_interruptions,status
0,cbe,dengue,2015-04-01,2016-02-01,10.0,2,recovered
1,cbe,dengue,2018-07-01,2019-04-01,9.0,2,recovered
2,cbe,dengue,2019-12-01,2020-06-01,6.0,0,recovered
3,cbe,dengue,2020-09-01,2020-12-01,3.0,0,recovered
4,cbe,dengue,2021-07-01,2021-11-01,4.0,0,recovered
5,cbe,dengue,2022-10-01,2023-04-01,6.0,0,recovered
6,cbe,dengue,2023-07-01,2023-10-01,3.0,0,recovered
7,cbe,dengue,2023-11-01,2024-02-01,3.0,0,recovered
8,cbe,malaria,2015-05-01,2015-12-01,7.0,1,recovered
9,cbe,malaria,2016-05-01,2016-10-01,5.0,0,recovered



Summary by disease:


,episodes,recovered,unresolved,mean_recovery_months,total_interruptions
disease_id,,,,,
dengue,39,38,1,6.55,30
malaria,53,50,3,5.28,29


In [3]:

#  Unit tests for the recovery detection engine

def make_test_series(flags):
    """True means cases exceed the seasonal threshold."""
    dates = pd.date_range(
        "2022-01-01",
        periods=len(flags),
        freq="MS"
    )

    return pd.DataFrame({
        "year_month": dates,
        "district_id": "test_district",
        "disease_id": "test_disease",
        "above_threshold": flags
    })


# Test 1: Normal sustained recovery
test1 = detect_recovery_episodes(
    make_test_series([True, True, False, False, False])
)

assert len(test1) == 1
assert test1[0]["status"] == "recovered"
assert test1[0]["recovery_months"] == 4
assert test1[0]["recovery_interruptions"] == 0

print("PASS: Normal sustained recovery")


# Test 2: Recovery interrupted by another outbreak wave
test2 = detect_recovery_episodes(
    make_test_series([
        True, True,
        False, False,
        True,
        False, False, False
    ])
)

assert len(test2) == 1
assert test2[0]["status"] == "recovered"
assert test2[0]["recovery_months"] == 7
assert test2[0]["recovery_interruptions"] == 1

print("PASS: Interrupted recovery")


# Test 3: Recovery not observed within dataset
test3 = detect_recovery_episodes(
    make_test_series([True, True, False, True, False])
)

assert len(test3) == 1
assert test3[0]["status"] == "unresolved"
assert pd.isna(test3[0]["recovery_months"])
assert test3[0]["recovery_interruptions"] == 1

print("PASS: Unresolved outbreak")


# Test 4: No outbreak
test4 = detect_recovery_episodes(
    make_test_series([False, False, False, False])
)

assert len(test4) == 0

print("PASS: No outbreak")


print("\nALL FOUR RECOVERY ENGINE TESTS PASSED")


PASS: Normal sustained recovery
PASS: Interrupted recovery
PASS: Unresolved outbreak
PASS: No outbreak

ALL FOUR RECOVERY ENGINE TESTS PASSED


In [4]:

# Validate monthly continuity and missing observations

def validate_recovery_data(data):
    required_columns = [
        "year_month",
        "district_id",
        "disease_id",
        "cases",
        "threshold_cases"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    validated = data.copy()

    validated["year_month"] = pd.to_datetime(
        validated["year_month"]
    )

    if validated["year_month"].isna().any():
        raise ValueError("Invalid or missing dates detected.")

    if not validated["year_month"].dt.is_month_start.all():
        raise ValueError(
            "year_month must contain first-of-month dates."
        )

    key_columns = [
        "district_id",
        "disease_id",
        "year_month"
    ]

    if validated[key_columns].isna().any().any():
        raise ValueError("Missing district, disease, or date keys.")

    if validated.duplicated(key_columns).any():
        raise ValueError(
            "Duplicate district-disease-month records detected."
        )

    if (
        (validated["cases"].dropna() < 0).any()
        or (validated["threshold_cases"].dropna() < 0).any()
    ):
        raise ValueError(
            "Negative case counts or thresholds detected."
        )

    validated = validated.sort_values(
        key_columns
    ).reset_index(drop=True)

    missing_months = []

    for (district, disease), group in validated.groupby(
        ["district_id", "disease_id"]
    ):
        expected_dates = pd.date_range(
            group["year_month"].min(),
            group["year_month"].max(),
            freq="MS"
        )

        missing_dates = expected_dates.difference(
            group["year_month"]
        )

        for date in missing_dates:
            missing_months.append({
                "district_id": district,
                "disease_id": disease,
                "missing_month": date
            })

    missing_values = validated[
        validated[["cases", "threshold_cases"]]
        .isna().any(axis=1)
    ]

    print("=== RECOVERY DATA QUALITY CHECK ===")
    print("Total records:", len(validated))
    print("Missing calendar months:", len(missing_months))
    print("Rows with missing cases/thresholds:", len(missing_values))

    return (
        validated,
        pd.DataFrame(missing_months),
        missing_values
    )


validated_recovery_data, missing_months, missing_values = (
    validate_recovery_data(recovery_data)
)

if len(missing_months) == 0 and len(missing_values) == 0:
    print("\nPASS: Recovery input data is complete.")
else:
    print("\nWARNING: Missing observations require handling.")
    display(missing_months.head())
    display(missing_values.head())


=== RECOVERY DATA QUALITY CHECK ===
Total records: 1200
Missing calendar months: 0
Rows with missing cases/thresholds: 0

PASS: Recovery input data is complete.


In [6]:

# Sustained recovery sensitivity analysis

assert missing_months.empty, "Missing calendar months detected."
assert missing_values.empty, "Missing case or threshold values detected."

sensitivity_results = []

for sustained_months in [2, 3, 4]:

    all_episodes = []

    for (district, disease), group in validated_recovery_data.groupby(
        ["district_id", "disease_id"]
    ):
        episodes = detect_recovery_episodes(
            group,
            sustained_months=sustained_months
        )

        all_episodes.extend(episodes)

    if not all_episodes:
        continue

    results = pd.DataFrame(all_episodes)

    summary = results.groupby("disease_id").agg(
        total_episodes=("status", "size"),
        recovered_episodes=(
            "status",
            lambda x: (x == "recovered").sum()
        ),
        unresolved_episodes=(
            "status",
            lambda x: (x == "unresolved").sum()
        ),
        mean_recovery_months=("recovery_months", "mean"),
        median_recovery_months=("recovery_months", "median"),
        total_interruptions=("recovery_interruptions", "sum")
    ).reset_index()

    summary["sustained_months"] = sustained_months

    sensitivity_results.append(summary)


sensitivity_summary = pd.concat(
    sensitivity_results,
    ignore_index=True
)

sensitivity_summary = sensitivity_summary[
    [
        "disease_id",
        "sustained_months",
        "total_episodes",
        "recovered_episodes",
        "unresolved_episodes",
        "mean_recovery_months",
        "median_recovery_months",
        "total_interruptions"
    ]
].sort_values(
    ["disease_id", "sustained_months"]
)

print("=== RECOVERY RULE SENSITIVITY ANALYSIS ===")

display(sensitivity_summary.round(2))


=== RECOVERY RULE SENSITIVITY ANALYSIS ===


,disease_id,sustained_months,total_episodes,recovered_episodes,unresolved_episodes,mean_recovery_months,median_recovery_months,total_interruptions
0,dengue,2,53,53,0,3.75,2.0,16
2,dengue,3,39,38,1,6.55,3.5,30
4,dengue,4,35,32,3,8.72,6.0,34
1,malaria,2,63,61,2,3.39,2.0,19
3,malaria,3,53,50,3,5.28,4.0,29
5,malaria,4,46,43,3,7.30,6.0,36


In [7]:

#  Prepare climate scenario recovery data

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent

cases = pd.read_csv(
    ROOT / "data/synthetic/disease_cases.csv",
    parse_dates=["year_month"]
)

thresholds = pd.read_csv(
    ROOT / "data/processed/recovery_thresholds.csv"
)

cases["calendar_month"] = cases["year_month"].dt.month
thresholds["calendar_month"] = thresholds["calendar_month"].astype(int)

# Only analyse the scenario evaluation period
scenario_cases = cases[
    cases["year_month"].between(
        "2022-01-01",
        "2024-12-01"
    )
].copy()

scenario_recovery = scenario_cases.merge(
    thresholds[
        ["district_id", "disease_id",
         "calendar_month", "threshold_cases"]
    ],
    on=["district_id", "disease_id", "calendar_month"],
    how="left",
    validate="many_to_one"
)

scenario_recovery["above_threshold"] = (
    scenario_recovery["cases"]
    > scenario_recovery["threshold_cases"]
)

scenario_recovery = scenario_recovery.sort_values(
    ["district_id", "disease_id",
     "scenario_id", "year_month"]
).reset_index(drop=True)

print("=== SCENARIO RECOVERY DATA ===")
print("Rows:", len(scenario_recovery))
print("Districts:", scenario_recovery["district_id"].nunique())
print("Diseases:", sorted(scenario_recovery["disease_id"].unique()))
print("Scenarios:", sorted(scenario_recovery["scenario_id"].unique()))
print(
    "Missing thresholds:",
    scenario_recovery["threshold_cases"].isna().sum()
)

print("\nRows per scenario:")
display(
    scenario_recovery.groupby("scenario_id")
    .size()
    .rename("rows")
    .reset_index()
)

print("\nSample:")
display(
    scenario_recovery[
        ["year_month", "district_id", "disease_id",
         "scenario_id", "cases", "threshold_cases",
         "above_threshold"]
    ].head(10)
)


=== SCENARIO RECOVERY DATA ===
Rows: 2520
Districts: 5
Diseases: ['dengue', 'malaria']
Scenarios: ['baseline', 'delayed_monsoon', 'heavy_rain_decline', 'irregular_rain', 'prolonged_wet', 'warm_wet', 'warmer']
Missing thresholds: 0

Rows per scenario:


,scenario_id,rows
0,baseline,360
1,delayed_monsoon,360
2,heavy_rain_decline,360
3,irregular_rain,360
4,prolonged_wet,360
5,warm_wet,360
6,warmer,360



Sample:


,year_month,district_id,disease_id,scenario_id,cases,threshold_cases,above_threshold
0,2022-01-01,cbe,dengue,baseline,112,127.2,False
1,2022-02-01,cbe,dengue,baseline,97,106.8,False
2,2022-03-01,cbe,dengue,baseline,80,81.6,False
3,2022-04-01,cbe,dengue,baseline,53,61.2,False
4,2022-05-01,cbe,dengue,baseline,48,48.0,False
5,2022-06-01,cbe,dengue,baseline,45,61.2,False
6,2022-07-01,cbe,dengue,baseline,60,73.2,False
7,2022-08-01,cbe,dengue,baseline,95,100.8,False
8,2022-09-01,cbe,dengue,baseline,79,109.2,False
9,2022-10-01,cbe,dengue,baseline,277,153.6,True
